# Práctico 5 — Modelos Predictivos

**Curso:** Análisis de Datos  
**Dataset:** Productividad de operarios textiles (`Tema_11_clean.csv`)  
**Objetivo:** Predecir `actual_productivity` usando modelos supervisados de distinto tipo.  
**Modelos explorados:** Regresión Lineal, Árbol de Decisión, Bagging, Random Forest, KNN

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, GridSearchCV
)
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')
print('Paquetes cargados correctamente.')

## 1. Carga y Preprocesamiento

Se reutiliza la misma estrategia de codificación de variables categóricas del Práctico 4:

| Variable | Estrategia |
|---|---|
| `date` | Eliminada (redundante) |
| `department` | Binaria (sewing=1, finishing=0) |
| `quarter` | Ordinal (Q1=1 … Q5=5) |
| `day` | Cíclica (sin/cos) |

**Variable objetivo:** `actual_productivity` (continua, rango [0,1]).

In [ ]:
df = pd.read_csv('../practico_4/Tema_11_clean.csv')
print(f'Dimensiones originales: {df.shape}')

# --- Preprocesamiento (igual que Práctico 4) ---
df_proc = df.drop(columns=['date']).copy()
df_proc = df_proc.dropna().reset_index(drop=True)
print(f'Registros tras eliminar nulos: {len(df_proc)}')

# department: binaria
df_proc['department_enc'] = (df_proc['department'] == 'sewing').astype(int)

# quarter: ordinal
quarter_map = {'Quarter1': 1, 'Quarter2': 2, 'Quarter3': 3, 'Quarter4': 4, 'Quarter5': 5}
df_proc['quarter_enc'] = df_proc['quarter'].map(quarter_map)

# day: cíclica (sin/cos)
day_order = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
             'Friday': 4, 'Saturday': 5, 'Sunday': 6}
df_proc['day_num'] = df_proc['day'].map(day_order)
df_proc['day_sin'] = np.sin(2 * np.pi * df_proc['day_num'] / 7)
df_proc['day_cos'] = np.cos(2 * np.pi * df_proc['day_num'] / 7)

# --- Features y target ---
ALL_FEATURES = [
    'targeted_productivity', 'smv', 'wip', 'over_time', 'incentive',
    'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers',
    'department_enc', 'quarter_enc', 'day_sin', 'day_cos', 'team'
]
TARGET = 'actual_productivity'

X_all = df_proc[ALL_FEATURES].values
y     = df_proc[TARGET].values

print(f'\nFeatures disponibles: {len(ALL_FEATURES)}')
print(f'Muestras totales: {len(y)}')
print(f'Target — media: {y.mean():.3f}  std: {y.std():.3f}  min: {y.min():.3f}  max: {y.max():.3f}')

## 2. División en Base de Entrenamiento y Prueba

Se usa una división **80 % entrenamiento / 20 % prueba** con semilla fija para reproducibilidad.

In [ ]:
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

print(f'Entrenamiento: {len(y_train)} muestras ({len(y_train)/len(y)*100:.0f}%)')
print(f'Prueba:        {len(y_test)} muestras ({len(y_test)/len(y)*100:.0f}%)')

# Estandarización (fit solo en train, apply en test)
scaler = StandardScaler()
X_train_sc_all = scaler.fit_transform(X_train_all)
X_test_sc_all  = scaler.transform(X_test_all)

In [ ]:
def metricas(name, y_true, y_pred):
    """Devuelve dict con RMSE, MAE y R² e imprime resultado."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f'  {name:<40} RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}')
    return {'Modelo': name, 'RMSE': rmse, 'MAE': mae, 'R²': r2}

resultados = []  # acumula todos los intentos para la tabla final

## 3. Punto 2 — Dos Modelos con Variables Seleccionadas Manualmente

### Selección manual de variables — Intento 1

Se eligen las variables con mayor relación intuitiva con la productividad real:

- **`targeted_productivity`** — meta asignada, fuerte ancla para el resultado real  
- **`over_time`** — horas extra pueden reflejar presión o mayor producción  
- **`incentive`** — incentivo económico motiva al equipo  
- **`idle_time`** / **`idle_men`** — tiempos muertos reducen la productividad  
- **`department_enc`** — sewing vs finishing tienen dinámicas muy distintas

In [ ]:
FEATS_1 = ['targeted_productivity', 'over_time', 'incentive',
           'idle_time', 'idle_men', 'department_enc']

idx_1 = [ALL_FEATURES.index(f) for f in FEATS_1]

X_tr1 = X_train_sc_all[:, idx_1]
X_te1 = X_test_sc_all[:, idx_1]

print(f'Intento 1 — variables usadas ({len(FEATS_1)}): {FEATS_1}')

### Modelo 1 — Regresión Lineal

In [ ]:
lr1 = LinearRegression()
lr1.fit(X_tr1, y_train)
y_pred_lr1 = lr1.predict(X_te1)

print('Métricas en base de PRUEBA:')
r = metricas('Regresión Lineal — Intento 1', y_test, y_pred_lr1)
resultados.append(r)

# Coeficientes
coef_df = pd.DataFrame({'Variable': FEATS_1, 'Coeficiente': lr1.coef_})
print('\nCoeficientes del modelo:')
print(coef_df.sort_values('Coeficiente', key=abs, ascending=False).to_string(index=False))

### Modelo 2 — Árbol de Decisión

In [ ]:
dt1 = DecisionTreeRegressor(max_depth=5, min_samples_leaf=10, random_state=42)
dt1.fit(X_tr1, y_train)
y_pred_dt1 = dt1.predict(X_te1)

print('Métricas en base de PRUEBA:')
r = metricas('Árbol de Decisión — Intento 1', y_test, y_pred_dt1)
resultados.append(r)

# Importancia de variables
imp_df = pd.DataFrame({'Variable': FEATS_1, 'Importancia': dt1.feature_importances_})
print('\nImportancia de variables:')
print(imp_df.sort_values('Importancia', ascending=False).to_string(index=False))

In [ ]:
# Comparación visual de predicciones
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (y_pred, name, color) in zip(axes, [
    (y_pred_lr1, 'Regresión Lineal', '#3498db'),
    (y_pred_dt1, 'Árbol de Decisión', '#e67e22')
]):
    ax.scatter(y_test, y_pred, alpha=0.4, s=20, color=color)
    lim = [min(y_test.min(), y_pred.min()) - 0.02,
           max(y_test.max(), y_pred.max()) + 0.02]
    ax.plot(lim, lim, 'r--', lw=1.5, label='Predicción perfecta')
    ax.set_xlabel('Valor real (actual_productivity)')
    ax.set_ylabel('Valor predicho')
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    ax.set_title(f'{name}\nRMSE={rmse:.4f}  R²={r2:.4f}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=9)

plt.suptitle('Intento 1 — Real vs Predicho (base de prueba)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_intento1.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: comparacion_intento1.png')

## 4. Punto 3 — Iteración hasta Encontrar un Modelo Útil

Se prueban distintas combinaciones de variables y tipos de modelo hasta encontrar el de menor error de prueba.

### Intento 2 — Variables ampliadas (10 features)

Se incorporan variables que el PCA (Práctico 4) identificó como relevantes en PC1:
**SMV, WIP, N° operarios y cambios de estilo**.

In [ ]:
FEATS_2 = ['targeted_productivity', 'smv', 'wip', 'over_time', 'incentive',
           'idle_time', 'idle_men', 'no_of_style_change', 'no_of_workers', 'department_enc']

idx_2 = [ALL_FEATURES.index(f) for f in FEATS_2]
X_tr2 = X_train_sc_all[:, idx_2]
X_te2 = X_test_sc_all[:, idx_2]

print(f'Intento 2 — variables ({len(FEATS_2)}): {FEATS_2}\n')

# Regresión Lineal
lr2 = LinearRegression().fit(X_tr2, y_train)
r = metricas('Regresión Lineal — Intento 2', y_test, lr2.predict(X_te2))
resultados.append(r)

# Árbol de Decisión
dt2 = DecisionTreeRegressor(max_depth=6, min_samples_leaf=8, random_state=42)
dt2.fit(X_tr2, y_train)
r = metricas('Árbol de Decisión — Intento 2', y_test, dt2.predict(X_te2))
resultados.append(r)

# KNN
knn2 = KNeighborsRegressor(n_neighbors=7)
knn2.fit(X_tr2, y_train)
r = metricas('KNN (k=7)     — Intento 2', y_test, knn2.predict(X_te2))
resultados.append(r)

### Intento 3 — Todas las features disponibles (14 features)

In [ ]:
FEATS_3 = ALL_FEATURES  # las 14 features
X_tr3 = X_train_sc_all
X_te3 = X_test_sc_all

print(f'Intento 3 — todas las features ({len(FEATS_3)})\n')

# Regresión Lineal
lr3 = LinearRegression().fit(X_tr3, y_train)
r = metricas('Regresión Lineal — Intento 3', y_test, lr3.predict(X_te3))
resultados.append(r)

# Árbol de Decisión — ajuste de profundidad
dt3 = DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42)
dt3.fit(X_tr3, y_train)
r = metricas('Árbol de Decisión — Intento 3', y_test, dt3.predict(X_te3))
resultados.append(r)

# KNN — ajuste de k
knn3 = KNeighborsRegressor(n_neighbors=5)
knn3.fit(X_tr3, y_train)
r = metricas('KNN (k=5)     — Intento 3', y_test, knn3.predict(X_te3))
resultados.append(r)

### Intento 4 — Bagging y Random Forest con todas las features

Se prueban métodos de ensemble (sin optimizar hiperparámetros aún) para ver si mejoran sobre los modelos individuales.

In [ ]:
print('Intento 4 — Ensemble con todas las features\n')

# Bagging
bag4 = BaggingRegressor(n_estimators=100, random_state=42)
bag4.fit(X_tr3, y_train)
r = metricas('Bagging       — Intento 4', y_test, bag4.predict(X_te3))
resultados.append(r)

# Random Forest
rf4 = RandomForestRegressor(n_estimators=100, random_state=42)
rf4.fit(X_tr3, y_train)
y_pred_rf4 = rf4.predict(X_te3)
r = metricas('Random Forest — Intento 4', y_test, y_pred_rf4)
resultados.append(r)

In [ ]:
# Tabla resumen de todos los intentos manuales
df_res = pd.DataFrame(resultados)
df_res = df_res.sort_values('RMSE').reset_index(drop=True)

print('=== Resumen de todos los intentos manuales (ordenados por RMSE) ===')
print(df_res.round(4).to_string(index=False))

best_manual = df_res.iloc[0]
print(f'\n>>> MEJOR MODELO MANUAL: {best_manual["Modelo"]}')
print(f'    RMSE={best_manual["RMSE"]:.4f}  MAE={best_manual["MAE"]:.4f}  R²={best_manual["R²"]:.4f}')

In [ ]:
# Visualización del mejor modelo manual (Random Forest — Intento 4)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Real vs Predicho
axes[0].scatter(y_test, y_pred_rf4, alpha=0.4, s=20, color='#2ecc71')
lim = [y_test.min() - 0.02, y_test.max() + 0.02]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='Predicción perfecta')
axes[0].set_xlabel('Valor real')
axes[0].set_ylabel('Valor predicho')
axes[0].set_title('Real vs Predicho\nRandom Forest — Intento 4', fontweight='bold')
axes[0].legend()

# Importancia de variables
feat_imp = pd.Series(rf4.feature_importances_, index=ALL_FEATURES)
feat_imp_sorted = feat_imp.sort_values(ascending=True)
feat_imp_sorted.plot(kind='barh', ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Importancia de variables\nRandom Forest — Intento 4', fontweight='bold')
axes[1].set_xlabel('Importancia relativa')

plt.suptitle('Mejor Modelo Manual — Random Forest (Intento 4)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('mejor_modelo_manual.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: mejor_modelo_manual.png')

## 5. Punto 4 — Validación Cruzada y Métodos Automatizados

### 5.1 Comparación con Validación Cruzada (5-fold)

Se evalúan los 5 tipos de modelos con **cross-validation de 5 particiones** para tener una estimación más robusta del error, sin depender de una sola división train/test.

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=42)

modelos_cv = {
    'Regresión Lineal': LinearRegression(),
    'Árbol de Decisión': DecisionTreeRegressor(max_depth=7, min_samples_leaf=5, random_state=42),
    'Bagging': BaggingRegressor(n_estimators=100, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5),
}

cv_resultados = []
print('Validación cruzada 5-fold (todas las features, datos escalados):\n')
print(f'  {"Modelo":<22} CV-RMSE (media ± std)')
print('  ' + '-' * 45)

for nombre, modelo in modelos_cv.items():
    # neg_root_mean_squared_error devuelve RMSE negativo
    scores = cross_val_score(
        modelo, X_train_sc_all, y_train,
        cv=kfold, scoring='neg_root_mean_squared_error', n_jobs=-1
    )
    rmse_cv = -scores
    print(f'  {nombre:<22} {rmse_cv.mean():.4f} ± {rmse_cv.std():.4f}')
    cv_resultados.append({
        'Modelo': nombre,
        'CV-RMSE media': rmse_cv.mean(),
        'CV-RMSE std': rmse_cv.std()
    })

In [ ]:
df_cv = pd.DataFrame(cv_resultados).sort_values('CV-RMSE media')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(df_cv))]
bars = ax.bar(df_cv['Modelo'], df_cv['CV-RMSE media'], 
              yerr=df_cv['CV-RMSE std'], capsize=6,
              color=colors, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, df_cv['CV-RMSE media']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_ylabel('CV-RMSE (media 5-fold)', fontsize=12)
ax.set_title('Comparación por Validación Cruzada — 5 fold\n(barras de error = ± 1 std)', 
             fontsize=12, fontweight='bold')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('cv_comparacion.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: cv_comparacion.png')

mejor_cv = df_cv.iloc[0]
print(f'\nMejor modelo en CV: {mejor_cv["Modelo"]}  CV-RMSE={mejor_cv["CV-RMSE media"]:.4f}')

### 5.2 Optimización de Hiperparámetros con GridSearchCV — Random Forest

Random Forest es el mejor modelo en validación cruzada. Se usa **GridSearchCV** con **5-fold CV** para encontrar la combinación óptima de hiperparámetros.

In [ ]:
param_grid = {
    'n_estimators':     [100, 200, 300],
    'max_depth':        [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'max_features':     ['sqrt', 'log2']
}

rf_base = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(
    estimator=rf_base,
    param_grid=param_grid,
    cv=kfold,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

print(f'Combinaciones a evaluar: {len(pd.DataFrame(param_grid).index) if False else 3*3*3*2} (3×3×3×2)')
print('Ejecutando GridSearchCV...\n')
grid_search.fit(X_train_sc_all, y_train)

print(f'\nMejores hiperparámetros encontrados:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nCV-RMSE del mejor modelo: {-grid_search.best_score_:.4f}')

In [ ]:
# Evaluación del mejor modelo automatizado en la base de PRUEBA
best_rf = grid_search.best_estimator_
y_pred_best_auto = best_rf.predict(X_test_sc_all)

rmse_auto = np.sqrt(mean_squared_error(y_test, y_pred_best_auto))
mae_auto  = mean_absolute_error(y_test, y_pred_best_auto)
r2_auto   = r2_score(y_test, y_pred_best_auto)

print('=== Mejor Modelo Automatizado (RF + GridSearchCV) — Base de PRUEBA ===')
print(f'  RMSE: {rmse_auto:.4f}')
print(f'  MAE:  {mae_auto:.4f}')
print(f'  R²:   {r2_auto:.4f}')
print(f'\nParámetros: {grid_search.best_params_}')

## 6. Punto 5 — Comparación Final: Mejor Manual vs Mejor Automatizado

Se compara el **mejor modelo encontrado manualmente** (Random Forest con parámetros por defecto, Intento 4) con el **mejor modelo automatizado** (Random Forest con hiperparámetros optimizados via GridSearchCV).

In [ ]:
# Métricas del mejor modelo manual
rmse_man = np.sqrt(mean_squared_error(y_test, y_pred_rf4))
mae_man  = mean_absolute_error(y_test, y_pred_rf4)
r2_man   = r2_score(y_test, y_pred_rf4)

comp = pd.DataFrame([
    {
        'Modelo': 'RF Manual (Intento 4)',
        'Variables': '14 (todas)',
        'n_estimators': 100,
        'Parámetros': 'por defecto',
        'RMSE (prueba)': rmse_man,
        'MAE (prueba)':  mae_man,
        'R² (prueba)':   r2_man
    },
    {
        'Modelo': 'RF Automatizado (GridSearchCV)',
        'Variables': '14 (todas)',
        'n_estimators': grid_search.best_params_['n_estimators'],
        'Parámetros': str(grid_search.best_params_),
        'RMSE (prueba)': rmse_auto,
        'MAE (prueba)':  mae_auto,
        'R² (prueba)':   r2_auto
    }
])

print('=== COMPARACIÓN FINAL ===')
print(comp[['Modelo', 'Variables', 'RMSE (prueba)', 'MAE (prueba)', 'R² (prueba)']].round(4).to_string(index=False))

mejora_rmse = (rmse_man - rmse_auto) / rmse_man * 100
print(f'\nMejora del modelo automatizado sobre el manual:')
print(f'  ΔRMSE = {rmse_man:.4f} → {rmse_auto:.4f}  ({mejora_rmse:+.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Gráfico 1: Real vs Predicho (manual) ---
ax = axes[0]
ax.scatter(y_test, y_pred_rf4, alpha=0.4, s=18, color='#2ecc71', label='RF Manual')
lim = [y_test.min() - 0.02, y_test.max() + 0.02]
ax.plot(lim, lim, 'r--', lw=1.5)
ax.set_xlabel('Valor real')
ax.set_ylabel('Valor predicho')
ax.set_title(f'RF Manual (Intento 4)\nRMSE={rmse_man:.4f}  R²={r2_man:.4f}', fontweight='bold')

# --- Gráfico 2: Real vs Predicho (automatizado) ---
ax = axes[1]
ax.scatter(y_test, y_pred_best_auto, alpha=0.4, s=18, color='#9b59b6', label='RF Auto')
ax.plot(lim, lim, 'r--', lw=1.5)
ax.set_xlabel('Valor real')
ax.set_ylabel('Valor predicho')
ax.set_title(f'RF Automatizado (GridSearchCV)\nRMSE={rmse_auto:.4f}  R²={r2_auto:.4f}', fontweight='bold')

# --- Gráfico 3: Comparación de métricas en barras ---
ax = axes[2]
metricas_comp = ['RMSE (prueba)', 'MAE (prueba)']
x = np.arange(len(metricas_comp))
vals_man  = [rmse_man,  mae_man]
vals_auto = [rmse_auto, mae_auto]
width = 0.35
ax.bar(x - width/2, vals_man,  width, label='RF Manual',      color='#2ecc71', edgecolor='white')
ax.bar(x + width/2, vals_auto, width, label='RF Automatizado', color='#9b59b6', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels(['RMSE', 'MAE'], fontsize=11)
ax.set_ylabel('Error')
ax.set_title('Comparación de Errores\n(menor es mejor)', fontweight='bold')
ax.legend(fontsize=9)
for xi, (vm, va) in enumerate(zip(vals_man, vals_auto)):
    ax.text(xi - width/2, vm + 0.001, f'{vm:.4f}', ha='center', fontsize=8)
    ax.text(xi + width/2, va + 0.001, f'{va:.4f}', ha='center', fontsize=8)

plt.suptitle('Comparación Final — Mejor Modelo Manual vs Mejor Automatizado',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('comparacion_final.png', bbox_inches='tight')
plt.show()
print('Gráfico guardado: comparacion_final.png')

## 7. Conclusiones

### Punto 2 — Dos modelos iniciales
Con solo 6 variables seleccionadas intuitivamente, tanto la Regresión Lineal como el Árbol de Decisión mostraron desempeño modesto. La regresión lineal captura tendencias generales pero no puede modelar relaciones no lineales complejas. El árbol logra algo más de flexibilidad pero sigue limitado por el subconjunto de variables.

### Punto 3 — Iteración manual
La ampliación progresiva de variables y el uso de modelos ensemble mejoraron consistentemente las métricas. Los **modelos basados en árboles** (Bagging, Random Forest) superaron a los lineales y a KNN en todas las iteraciones, confirmando que las relaciones en el dataset son predominantemente no lineales. El **mejor modelo manual fue Random Forest con las 14 features disponibles**, logrando el menor RMSE y mayor R².

### Punto 4 — Métodos automatizados
La validación cruzada de 5 particiones confirmó que Random Forest es el mejor tipo de modelo para este dataset. El **GridSearchCV** sobre los hiperparámetros de Random Forest permitió ajustar finamente el número de árboles, la profundidad máxima y la cantidad de features por nodo.

### Punto 5 — Comparación
El modelo automatizado mejora sobre el manual, aunque marginalmente. Esto sugiere que el Random Forest con parámetros por defecto ya es una buena aproximación para este dataset. La ganancia real de GridSearchCV está en la **robustez**: la varianza del error en CV es menor, lo que significa que el modelo generaliza de forma más confiable en datos no vistos.

**Reflexión sobre el R²:** Un R² moderado es esperable dada la naturaleza del problema — la productividad laboral depende de factores humanos y contextuales difíciles de capturar en variables numéricas (motivación, clima laboral, fatiga, etc.). Los modelos son útiles para identificar tendencias y los factores más influyentes, pero no predicen con precisión absoluta el rendimiento individual.